# Bölüm 3 — Uygulama: Risk Segmentasyonu ve Belirsiz Kayıtların Sınıflandırılması

**Proje:** GPT Eklenti Ekosisteminde Gizlilik Riski Analizi ve Otomatik Sınıflandırma

**Araştırma Sorusu 4:** Eklentileri topladıkları veri türüne göre gruplandırdığımızda doğal riskli/risksiz kümeler ortaya çıkıyor mu? (kümeleme)

**Araştırma Sorusu 5:** Bölüm 2'de kurduğumuz model, etiketi belirsiz olan kayıtları (özellikle `data_type`'ta "Other" kategorisi, 3.544 kayıt) sınıflandırmak için kullanılabilir mi?

**Bu bölümün planı:**
1. Problemi kurma: parametre-seviyesinden eklenti-seviyesine geçiş — her eklentinin "veri toplama profilini" çıkarma
2. Kümeleme: eklentileri veri toplama profiline göre gruplama, kümeleri yorumlama (riskli/risksiz ayrımı var mı?)
3. Bölüm 2 modelini "Other" kayıtlarına uygulama, sonuçları temkinli yorumlama
4. Genel proje özeti

## Adım 1 — Parametre Seviyesinden Eklenti Seviyesine Geçiş

Bölüm 1 ve 2'de her satır bir **parametreydi**. Ama Araştırma Sorusu 4 eklentileri (GPT Actions) gruplamak istiyor — yani analiz birimimizi değiştirmemiz lazım: parametreden **eklentiye**.

**Zorluk:** `plugin_id_filenames` bir liste — aynı parametre birden fazla eklenti tarafından kullanılabiliyor (örn. yaygın bir "user_id" parametresi yüzlerce eklentide geçebilir). Bunu çözmek için veriyi "patlatacağız" (explode): her (eklenti, parametre) çifti kendi satırı olacak. Böylece "bu eklenti hangi parametreleri/kategorileri topluyor" sorusuna cevap verebileceğiz.

**Eklenti profili nasıl çıkarılır:** Her eklenti için, topladığı parametrelerin **kategori bazında oranını** hesaplayacağız (örn. "bu eklentinin parametrelerinin %30'u Identifier, %10'u Security credentials..."). Oran kullanıyoruz, ham sayı değil — çünkü eklentiler büyüklük olarak çok farklı (1 parametreden 1000+ parametreye kadar); önemli olan eklentinin büyüklüğü değil, *neyi topladığının profili*.

**Minimum parametre eşiği:** Sadece 1-2 parametresi olan bir eklentinin "profili" anlamsız olur (tek kategoriye %100 ait görünür, bu bir örüntü değil şans eseri). Bu yüzden en az **3 parametreli** eklentileri alacağız — bu bize 4.592 eklentiden **3.041**'ini bırakıyor, yeterince büyük bir örneklem.

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

DATA_PATH = '../backend/data_entries_final.json'

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)
df['text'] = (df['name'].fillna('') + '. ' + df['description'].fillna('')).str.strip()

df_exploded = df.explode('plugin_id_filenames').rename(columns={'plugin_id_filenames': 'plugin_id'}).reset_index(drop=True)

print('Parametre kaydı sayısı:', len(df))
print('Patlatma sonrası (eklenti, parametre) satır sayısı:', len(df_exploded))
print('Benzersiz eklenti sayısı:', df_exploded['plugin_id'].nunique())

Parametre kaydı sayısı: 12811
Patlatma sonrası (eklenti, parametre) satır sayısı: 40261
Benzersiz eklenti sayısı: 4592


In [2]:
MIN_PARAMS = 3

plugin_param_counts = df_exploded.groupby('plugin_id').size()
eligible_plugins = plugin_param_counts[plugin_param_counts >= MIN_PARAMS].index

df_exploded_filtered = df_exploded[df_exploded['plugin_id'].isin(eligible_plugins)]

plugin_category_counts = pd.crosstab(df_exploded_filtered['plugin_id'], df_exploded_filtered['main_data_type'])
plugin_profile = plugin_category_counts.div(plugin_category_counts.sum(axis=1), axis=0)

print(f'{MIN_PARAMS}+ parametreli eklenti sayısı: {len(plugin_profile)}')
print('Profil matrisi boyutu:', plugin_profile.shape)
plugin_profile.head(3)

3+ parametreli eklenti sayısı: 3041
Profil matrisi boyutu: (3041, 25)


main_data_type,App metadata,App usage data,E-commerce data,Event information,Files and documents,Finance information,Food and nutrition information,Gaming data,Health information,Identifier,Legal and law enforcement data,Location,Market data,Message,Other,Personal information,Query,Real estate data,Security credentials,Sports information,Time,Travel information,Vehicle information,Weather information,Web and network data
plugin_id,,,,,,,,,,,,,,,,,,,,,,,,,
g-009xFjOLC_OoDqO0qGP1tw2zGRYCYIfpUK,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0
g-00y8F1eRb_APoRCDy5ffgP0bHoQMuTHeML,0.00,0.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.00,0.2,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.1
g-02IWjW9i4_gzm_cnf_brjSH6N13PZzuSsTMwtje1dt~gzm_tool_u3iT5JbQPY0dX8WI6dUjMMYB,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,1.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
